ACTIVIDAD 1: Exploración de datos.

In [1]:
import pandas as pd
import numpy as np

# Cargar datos
df = pd.read_csv("Datos_Laboratorio3.csv", sep=";", encoding="latin1")
df_test = pd.read_csv("Datos test Lab3.csv", sep=";", encoding="latin1")

print("Shape train:", df.shape)
print("Shape test :", df_test.shape)

print("\nColumnas:")
print(df.columns.tolist())

print("\nTipos de dato:")
print(df.dtypes)

print("\nValores faltantes train:")
print(df.isna().sum().sort_values(ascending=False))

print("\nValores faltantes test:")
print(df_test.isna().sum().sort_values(ascending=False))

print("\nDuplicados en train:", df.duplicated().sum())
print("Duplicados en test :", df_test.duplicated().sum())

print("\nDistribución Plan_entrenamiento:")
print(df["Plan_entrenamiento"].value_counts())
print(df["Plan_entrenamiento"].value_counts(normalize=True))

print("\nDistribución Plan_nutrición:")
print(df["Plan_nutrición"].value_counts())
print(df["Plan_nutrición"].value_counts(normalize=True))

Shape train: (9698, 26)
Shape test : (302, 26)

Columnas:
['Edad', 'Gnereo', 'Peso', 'Altura', 'BMI', 'Objetivo', 'Condicion_salud', 'Nivel_Actividad', 'Nivel_experiencia', 'Dieta_preferida', 'Horas_sueño', 'Entrenamiento_preferido', 'Cantidad_equipo', 'Tiempo_disponible', 'Tiene_alergia', 'Problemas_digestivos', 'Fumador', 'Cigarrillos_dia', 'Alcohol', 'Alcohol_semana', 'Score_micronutrientes', 'Ingesta_proteinas', 'Pasos_dia', 'Ingesta_agua', 'Plan_entrenamiento', 'Plan_nutrición']

Tipos de dato:
Edad                         int64
Gnereo                      object
Peso                       float64
Altura                     float64
BMI                        float64
Objetivo                    object
Condicion_salud             object
Nivel_Actividad             object
Nivel_experiencia           object
Dieta_preferida             object
Horas_sueño                float64
Entrenamiento_preferido     object
Cantidad_equipo              int64
Tiempo_disponible            int64
Tiene

El conjunto de entrenamiento contiene 9698 observaciones y 26 variables, mientras que el conjunto de prueba contiene 302 observaciones con la misma estructura, aunque sin las variables objetivo. Esto confirma que el problema corresponde a una tarea de clasificación supervisada, donde se deben predecir las variables Plan_entrenamiento y Plan_nutrición.

En cuanto a los tipos de datos, se identifican tres categorías principales:

Variables numéricas: Edad, Peso, Altura, BMI, Horas_sueño, Cantidad_equipo, Tiempo_disponible, Cigarrillos_dia.
Variables categóricas: Gnereo, Objetivo, Condicion_salud, Nivel_Actividad, Nivel_experiencia, Dieta_preferida, Entrenamiento_preferido.
Variables binarias (0/1): Tiene_alergia, Problemas_digestivos, Fumador.

Se detecta un problema de calidad en el nombre de la variable Gnereo, que probablemente corresponde a Genero, lo cual deberá corregirse durante la etapa de limpieza.

Adicionalmente, se identifican valores faltantes en la variable Peso, lo que indica la necesidad de aplicar una estrategia de imputación en la fase de preprocesamiento.

En relación con la variable objetivo Plan_entrenamiento, se observa un desbalance moderado de clases, donde:

Sin plan: ~26%
Basico: ~23.6%
Especializado: ~6.1%

Esto sugiere que algunas clases están menos representadas, lo cual puede afectar el rendimiento del modelo. Por esta razón, será importante utilizar métricas como F1-score y aplicar técnicas como validación cruzada para una evaluación más robusta.

Finalmente, no se evidencian problemas estructurales graves en los datos, pero sí se requiere un adecuado proceso de limpieza, codificación de variables categóricas e imputación de valores faltantes antes de proceder con el modelado.

ACTIVIDAD 2: Propuesta de limpieza y preparacion de los datos

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# 1. Correcciones básicas

df = df.rename(columns={"Gnereo": "Genero"})
df_test = df_test.rename(columns={"Gnereo": "Genero"})

# 2. Definir variables

target_entrenamiento = "Plan_entrenamiento"
target_nutricion = "Plan_nutrición"

X = df.drop(columns=[target_entrenamiento, target_nutricion])
y_train_entrenamiento = df[target_entrenamiento]
y_train_nutricion = df[target_nutricion]

# 3. Separar tipos de variables

numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numéricas:", numerical_cols)
print("Categóricas:", categorical_cols)

# 4. Pipelines de transformación

# Numéricas: imputar + escalar
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categóricas: imputar + one-hot
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 5. ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# 6. Train/Test split

X_train, X_val, y_train_ent, y_val_ent = train_test_split(
    X, y_train_entrenamiento,
    test_size=0.2,
    random_state=42,
    stratify=y_train_entrenamiento
)

_, _, y_train_nut, y_val_nut = train_test_split(
    X, y_train_nutricion,
    test_size=0.2,
    random_state=42,
    stratify=y_train_nutricion
)

print("Split listo:")
print("Train:", X_train.shape)
print("Val  :", X_val.shape)

Numéricas: ['Edad', 'Peso', 'Altura', 'BMI', 'Horas_sueño', 'Cantidad_equipo', 'Tiempo_disponible', 'Tiene_alergia', 'Problemas_digestivos', 'Fumador', 'Cigarrillos_dia', 'Alcohol', 'Alcohol_semana', 'Score_micronutrientes', 'Ingesta_proteinas', 'Pasos_dia', 'Ingesta_agua']
Categóricas: ['Genero', 'Objetivo', 'Condicion_salud', 'Nivel_Actividad', 'Nivel_experiencia', 'Dieta_preferida', 'Entrenamiento_preferido']
Split listo:
Train: (7758, 24)
Val  : (1940, 24)


A partir del análisis exploratorio, se identificaron varios aspectos que requieren tratamiento antes de construir los modelos:

Se corrigió el nombre de la variable Gnereo a Genero para garantizar consistencia en los nombres de las variables.
Se identificaron valores faltantes en la variable Peso, por lo que se aplicará imputación utilizando la mediana, al ser una estrategia robusta frente a valores atípicos.
Se separaron las variables en dos grupos principales:
Variables numéricas: Edad, Peso, Altura, BMI, Horas_sueño, Cantidad_equipo, Tiempo_disponible, Tiene_alergia, Problemas_digestivos, entre otras.
Variables categóricas: Genero, Objetivo, Condicion_salud, Nivel_Actividad, Nivel_experiencia, Dieta_preferida, Entrenamiento_preferido.
Las variables numéricas serán escaladas mediante StandardScaler, mientras que las categóricas serán transformadas mediante One-Hot Encoding.
Se implementó un pipeline de preprocesamiento utilizando ColumnTransformer, lo que permite integrar todas las transformaciones en un flujo reproducible y evita problemas de fuga de información (data leakage).

Posteriormente, se realizó una partición del conjunto de datos en entrenamiento y validación utilizando un 80% para entrenamiento y 20% para validación, aplicando estratificación sobre la variable objetivo para preservar la distribución de clases. Como resultado:

Conjunto de entrenamiento: 7758 observaciones
Conjunto de validación: 1940 observaciones

Finalmente, este preprocesamiento será integrado directamente en los modelos mediante pipelines, asegurando consistencia entre las etapas de entrenamiento y predicción.

Cabe resaltar que el uso de pipelines garantiza que las transformaciones aprendidas en los datos de entrenamiento se apliquen de manera consistente sobre los datos de validación y prueba, evitando fugas de información y asegurando la reproducibilidad del proceso.

ACTIVIDAD 3: Desarrollar 2 modelos de regresión logística

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# PIPELINE COMPLETO

pipe_logistic = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# GRID DE HIPERPARÁMETROS

param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l2"],  # l1 solo con liblinear
    "model__solver": ["lbfgs", "liblinear"]
}

# MODELO 1: ENTRENAMIENTO

grid_entrenamiento = GridSearchCV(
    pipe_logistic,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_entrenamiento.fit(X_train, y_train_ent)

print("Mejores parámetros (Entrenamiento):")
print(grid_entrenamiento.best_params_)

y_pred_ent = grid_entrenamiento.predict(X_val)

print("\nReporte - Plan Entrenamiento:")
print(classification_report(y_val_ent, y_pred_ent))

Mejores parámetros (Entrenamiento):
{'model__C': 10, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}

Reporte - Plan Entrenamiento:
              precision    recall  f1-score   support

        Alto       0.89      0.89      0.89       131
        Bajo       0.82      0.79      0.81       458
       Medio       0.89      0.90      0.90       589
     Ninguno       0.94      0.96      0.95       762

    accuracy                           0.90      1940
   macro avg       0.89      0.88      0.88      1940
weighted avg       0.89      0.90      0.89      1940



In [4]:
# MODELO 2: NUTRICIÓN

grid_nutricion = GridSearchCV(
    pipe_logistic,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_nutricion.fit(X_train, y_train_nut)

print("Mejores parámetros (Nutrición):")
print(grid_nutricion.best_params_)

y_pred_nut = grid_nutricion.predict(X_val)

print("\nReporte - Plan Nutrición:")
print(classification_report(y_val_nut, y_pred_nut))

Mejores parámetros (Nutrición):
{'model__C': 1, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}

Reporte - Plan Nutrición:
               precision    recall  f1-score   support

   Balanceado       0.44      0.99      0.61       860
       Basico       0.00      0.00      0.00       457
Especializado       0.00      0.00      0.00       118
     Sin plan       0.38      0.01      0.02       505

     accuracy                           0.44      1940
    macro avg       0.21      0.25      0.16      1940
 weighted avg       0.30      0.44      0.28      1940



c:\Users\Daniel\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Daniel\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Daniel\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

El modelo de regresión logística para la predicción de Plan_entrenamiento presenta un buen desempeño general, con un accuracy cercano a 0.90 y valores equilibrados de precision, recall y F1-score en todas las clases. Esto indica que el modelo logra capturar adecuadamente los patrones presentes en los datos para este objetivo.

Por otro lado, el modelo para Plan_nutrición presenta un desempeño significativamente inferior. Se observa que el modelo tiende a predecir casi exclusivamente la clase "Balanceado", ignorando en gran medida las demás clases como "Basico", "Especializado" y "Sin plan".

Esto se refleja en:

Valores de recall cercanos a 0 en varias clases.
F1-score igual a 0 en clases no predichas.
Aparición de advertencias (UndefinedMetricWarning), lo cual indica que algunas clases no fueron predichas por el modelo.

Este comportamiento sugiere un problema de desbalance de clases, donde el modelo favorece la clase mayoritaria, afectando negativamente su capacidad de generalización para clases menos representadas.

En consecuencia, el modelo de regresión logística resulta adecuado como línea base, pero no es suficiente para capturar correctamente la complejidad del problema de nutrición, por lo que será necesario evaluar modelos más flexibles, como los árboles de decisión.

ACTIVIDAD 4: Desarrollar 2 modelos basados en árboles de decision

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Pipeline con preprocesamiento + árbol
pipe_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

# Grid de hiperparámetros
param_grid_tree = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 5, 10, 15, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10]
}

# Modelo para Plan_entrenamiento
grid_tree_ent = GridSearchCV(
    estimator=pipe_tree,
    param_grid=param_grid_tree,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_tree_ent.fit(X_train, y_train_ent)

print("Mejores parámetros (Árbol - Entrenamiento):")
print(grid_tree_ent.best_params_)

y_pred_tree_ent = grid_tree_ent.predict(X_val)

print("\nReporte - Árbol Plan Entrenamiento:")
print(classification_report(y_val_ent, y_pred_tree_ent, zero_division=0))

Mejores parámetros (Árbol - Entrenamiento):
{'model__criterion': 'entropy', 'model__max_depth': 5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}

Reporte - Árbol Plan Entrenamiento:
              precision    recall  f1-score   support

        Alto       1.00      1.00      1.00       131
        Bajo       1.00      1.00      1.00       458
       Medio       1.00      1.00      1.00       589
     Ninguno       1.00      1.00      1.00       762

    accuracy                           1.00      1940
   macro avg       1.00      1.00      1.00      1940
weighted avg       1.00      1.00      1.00      1940



In [6]:
# Modelo para Plan_nutrición
grid_tree_nut = GridSearchCV(
    estimator=pipe_tree,
    param_grid=param_grid_tree,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_tree_nut.fit(X_train, y_train_nut)

print("Mejores parámetros (Árbol - Nutrición):")
print(grid_tree_nut.best_params_)

y_pred_tree_nut = grid_tree_nut.predict(X_val)

print("\nReporte - Árbol Plan Nutrición:")
print(classification_report(y_val_nut, y_pred_tree_nut, zero_division=0))

Mejores parámetros (Árbol - Nutrición):
{'model__criterion': 'gini', 'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5}

Reporte - Árbol Plan Nutrición:
               precision    recall  f1-score   support

   Balanceado       0.46      0.48      0.47       860
       Basico       0.22      0.24      0.23       457
Especializado       0.12      0.09      0.10       118
     Sin plan       0.28      0.25      0.27       505

     accuracy                           0.34      1940
    macro avg       0.27      0.27      0.27      1940
 weighted avg       0.33      0.34      0.34      1940



Para la variable objetivo Plan_entrenamiento, el modelo de árbol de decisión obtuvo un desempeño perfecto en el conjunto de validación, alcanzando valores de precisión, recall y F1-score iguales a 1.00 en todas las clases.

Si bien este resultado puede parecer ideal, es importante considerar que este comportamiento puede ser indicativo de sobreajuste (overfitting), donde el modelo ha aprendido patrones demasiado específicos del conjunto de entrenamiento, lo que podría afectar su capacidad de generalización a nuevos datos.

En contraste, para la variable Plan_nutrición, el modelo de árbol de decisión mostró una mejora significativa frente a la regresión logística. A diferencia del modelo anterior, el árbol logra predecir todas las clases, evitando el problema de colapso hacia una sola categoría.

Sin embargo, el desempeño general sigue siendo limitado:

F1-score macro cercano a 0.27
Dificultad para predecir correctamente clases como "Especializado" y "Basico"
Mejor desempeño en la clase mayoritaria "Balanceado"

Esto sugiere que, aunque el árbol de decisión es más flexible y captura mejor la estructura de los datos, el problema de predicción del plan de nutrición sigue siendo complejo, posiblemente debido a:

Desbalance de clases
Baja separabilidad entre categorías
Variables insuficientemente informativas

En general, el árbol de decisión representa una mejora frente a la regresión logística para el caso de nutrición, pero aún presenta limitaciones importantes en su desempeño.

ACTIVIDAD 5: Tabla Comparativa

In [7]:
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

def metrics(y_true, y_pred):
    return {
        "Precision": precision_score(y_true, y_pred, average="macro"),
        "Recall": recall_score(y_true, y_pred, average="macro"),
        "F1-score": f1_score(y_true, y_pred, average="macro")
    }

results = pd.DataFrame([
    ["Logística", "Entrenamiento", *metrics(y_val_ent, y_pred_ent).values()],
    ["Árbol", "Entrenamiento", *metrics(y_val_ent, y_pred_tree_ent).values()],
    ["Logística", "Nutrición", *metrics(y_val_nut, y_pred_nut).values()],
    ["Árbol", "Nutrición", *metrics(y_val_nut, y_pred_tree_nut).values()],
], columns=["Modelo", "Objetivo", "Precision", "Recall", "F1-score"])

results

c:\Users\Daniel\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,Modelo,Objetivo,Precision,Recall,F1-score
0,Logística,Entrenamiento,0.886782,0.882910,0.884730
1,Árbol,Entrenamiento,1.000000,1.000000,1.000000
2,Logística,Nutrición,0.207019,0.249859,0.157939
3,Árbol,Nutrición,0.268462,0.265025,0.266059
